In [8]:
import transformers

print("transformers version:", transformers.__version__)


transformers version: 5.15.1


In [9]:
import torch

print("torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU actual:", torch.cuda.current_device())
    print("Nombre de la GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
    x = torch.randn(2, 3, device="cuda")
    print("Prueba de tensor en GPU:", x)
else:
    print("No hay CUDA disponible. PyTorch está usando CPU.")

"""Verifica el dispositivo antes de ejecutar la celda de Qwen.
Si la siguiente celda del modelo muestra 'cuda:0', entonces la GPU está activa."""


torch version: 2.13.0+cu130
CUDA disponible: True
Número de GPUs: 8
GPU actual: 0
Nombre de la GPU: NVIDIA A16
Prueba de tensor en GPU: tensor([[-0.0592, -0.0325,  0.5567],
        [-0.7360,  0.0079,  1.8176]], device='cuda:0')


"Verifica el dispositivo antes de ejecutar la celda de Qwen.\nSi la siguiente celda del modelo muestra 'cuda:0', entonces la GPU está activa."

In [15]:
%%time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook requires a GPU.")

device = torch.device("cuda:0")
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-3B-Instruct"
print("Using GPU:", torch.cuda.get_device_name(0))

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

print("model device:", next(model.parameters()).device)
print("input device check will follow below")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print("input_ids device:", model_inputs["input_ids"].device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Using GPU: NVIDIA A16


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 298.94it/s]


model device: cuda:0
input device check will follow below
input_ids device: cuda:0
A large language model (LLM) is a type of artificial intelligence model that can understand and generate human-like text based on vast amounts of training data. These models are trained using advanced machine learning techniques, often involving deep neural networks, to learn patterns and structures in natural language.

Key characteristics of large language models include:

1. **Scale**: They are typically trained on enormous datasets, sometimes reaching petabytes or even exabytes in size. This extensive training helps the model understand and produce a wide range of linguistic variations and contexts.

2. **Contextual Understanding**: Modern large language models are capable of understanding context, allowing them to generate coherent responses to complex questions or prompts that involve multiple layers of meaning.

3. **Multilingualism**: Many large language models are designed to be multilingual, ca

In [17]:
%%time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook requires a GPU.")

device = torch.device("cuda:4")
torch.cuda.empty_cache()

model_name = "google/gemma-3-1b-it"
print("Using GPU:", torch.cuda.get_device_name(4))

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 4},
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

print("model device:", next(model.parameters()).device)
print("input device check will follow below")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print("input_ids device:", model_inputs["input_ids"].device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Using GPU: NVIDIA A16


Loading weights: 100%|██████████| 340/340 [00:00<00:00, 544.94it/s]


model device: cuda:4
input device check will follow below
input_ids device: cuda:4
Okay, here’s a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a revolutionary type of artificial intelligence that are rapidly changing how we interact with computers.**

Basically, they’re incredibly complex computer programs trained on massive amounts of text and code. Think of them as digital parrots – they’ve learned to mimic human language patterns by reading billions of words from the internet, books, and more.

**Here’s a breakdown of what they do:**

*   **They can:**
    *   Generate text: Write articles, poems, code, scripts, emails, etc.
    *   Translate languages.
    *   Answer your questions in an informative way.
    *   Summarize text.
    *   And much more!

*   **How they work:** They use a technique called "deep learning" – a complex set of mathematical models – to predict the next word in a sequence.  The more data they’re trained on, the bett

In [18]:
def gpu_memory(device_id):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3
    total = torch.cuda.get_device_properties(device_id).total_memory / 1024**3
    print(
        f"cuda:{device_id} | "
        f"allocated: {allocated:.2f} GiB | "
        f"reserved: {reserved:.2f} GiB | "
        f"total: {total:.2f} GiB"
    )

gpu_memory(0)
gpu_memory(4)

cuda:0 | allocated: 0.01 GiB | reserved: 5.78 GiB | total: 14.61 GiB
cuda:4 | allocated: 1.87 GiB | reserved: 5.79 GiB | total: 14.61 GiB


# Releasing resources
Once finished we shall release the resource since PyTorch allocates the model in memory so it can be used from cache and inference be faster

In [ ]:
import gc
import torch

# Remove Python references to GPU-resident objects
for name in (
    "model",
    "tokenizer",
    "model_inputs",
    "generated_ids",
    "response",
    "text",
    "messages",
):
    globals().pop(name, None)

# Force Python garbage collection, then release unused PyTorch cache
gc.collect()

for device_id in range(torch.cuda.device_count()):
    with torch.cuda.device(device_id):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()